# Colony Detector — Training Notebook

Trains a YOLO object detector to find colonies on your plate photos, using the
dataset you built with the labeller.

**Before you start:** in Colab, turn on the free GPU —
`Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save`.

Then run the cells top to bottom. When it asks you to upload, give it the
`data` folder from your labeller, zipped.

To make that zip on Windows: go to `colony-counter\labeller\`, right-click the
`data` folder → *Send to → Compressed (zipped) folder*. That gives `data.zip`.


## 1. Check the GPU and install YOLO

In [ ]:
# Confirm a GPU is attached (you should see a Tesla T4 or similar).
!nvidia-smi -L

# Install Ultralytics YOLO.
%pip install -q ultralytics
import ultralytics
ultralytics.checks()


## 2. Upload your dataset

Run this cell, click **Choose Files**, and pick the `data.zip` you made from
the labeller's `data` folder. It unzips and sanitises filenames (spaces and
parentheses in names can trip up training, so we rename copies safely).


In [ ]:
import os, zipfile, shutil, glob, re
from google.colab import files

# Clean any previous run.
for p in ['data.zip', 'data', 'dataset']:
    if os.path.isfile(p): os.remove(p)
    elif os.path.isdir(p): shutil.rmtree(p)

print("Choose your data.zip …")
uploaded = files.upload()

zip_name = next((k for k in uploaded if k.lower().endswith('.zip')), None)
assert zip_name, "No .zip uploaded — please upload the zipped data folder."

with zipfile.ZipFile(zip_name) as z:
    z.extractall('unzipped')

# The zip may contain data/images or just images/ — find the images+labels dirs.
def find_dir(target):
    for root, dirs, _ in os.walk('unzipped'):
        if os.path.basename(root) == target and glob.glob(os.path.join(root, '*')):
            return root
    return None

img_src = find_dir('images')
lbl_src = find_dir('labels')
assert img_src and lbl_src, "Couldn't find images/ and labels/ in the zip."
print("images from:", img_src)
print("labels from:", lbl_src)


## 3. Sanitise filenames and pair images with labels

In [ ]:
# Build a clean dataset with safe filenames (no spaces/parentheses),
# keeping each image matched to its label file.
os.makedirs('dataset/images', exist_ok=True)
os.makedirs('dataset/labels', exist_ok=True)

def safe(name):
    stem = os.path.splitext(name)[0]
    stem = re.sub(r'[^A-Za-z0-9_-]+', '_', stem).strip('_')
    return stem or 'plate'

pairs = 0
used = set()
for img_path in sorted(glob.glob(os.path.join(img_src, '*'))):
    ext = os.path.splitext(img_path)[1].lower()
    if ext not in ('.jpg', '.jpeg', '.png'):
        continue
    stem = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = os.path.join(lbl_src, stem + '.txt')
    if not os.path.exists(lbl_path):
        continue  # skip images without a label file
    clean = safe(os.path.basename(img_path))
    n = clean; i = 1
    while n in used:
        n = f"{clean}_{i}"; i += 1
    used.add(n)
    shutil.copy(img_path, f'dataset/images/{n}.jpg')
    shutil.copy(lbl_path, f'dataset/labels/{n}.txt')
    pairs += 1

print(f"Paired {pairs} image+label sets.")
assert pairs > 0, "No matched image/label pairs found."


## 4. Split into training and validation sets

We hold out ~20% of the plates the model never trains on, so we can judge
whether it actually learned to detect colonies rather than just memorising.


In [ ]:
import random
random.seed(42)

os.makedirs('dataset/train/images', exist_ok=True)
os.makedirs('dataset/train/labels', exist_ok=True)
os.makedirs('dataset/val/images', exist_ok=True)
os.makedirs('dataset/val/labels', exist_ok=True)

stems = [os.path.splitext(os.path.basename(p))[0]
         for p in glob.glob('dataset/images/*.jpg')]
random.shuffle(stems)
n_val = max(1, int(len(stems) * 0.2))
val = set(stems[:n_val])

for s in stems:
    split = 'val' if s in val else 'train'
    shutil.copy(f'dataset/images/{s}.jpg', f'dataset/{split}/images/{s}.jpg')
    shutil.copy(f'dataset/labels/{s}.txt', f'dataset/{split}/labels/{s}.txt')

print(f"Train: {len(stems)-len(val)} plates   Validation: {len(val)} plates")


## 5. Write the dataset config

In [ ]:
# Read class names from the labeller if present, else default to one class.
class_names = ['colony']
for cand in glob.glob('unzipped/**/classes.txt', recursive=True):
    with open(cand) as f:
        names = [l.strip() for l in f if l.strip()]
    if names:
        class_names = names
    break

yaml = f'''path: {os.path.abspath('dataset')}
train: train/images
val: val/images
names:
'''
for i, name in enumerate(class_names):
    yaml += f'  {i}: {name}\n'

with open('dataset/data.yaml', 'w') as f:
    f.write(yaml)
print(yaml)
print("Classes:", class_names)


## 6. Train

Starts from a small pretrained YOLO model and fine-tunes it on your plates.
On a T4 GPU this is minutes, not hours. `imgsz=1024` keeps enough detail to
see small colonies; drop to 640 if you hit memory limits.

If your dataset is small, more `epochs` helps up to a point — 100 is a
reasonable start with early-stopping (`patience`) so it won't overtrain.


In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')   # nano: fast, resists overfitting on small data

results = model.train(
    data='dataset/data.yaml',
    epochs=100,
    imgsz=1024,
    batch=8,
    patience=25,             # stop if val stops improving for 25 epochs
    project='runs',
    name='colony',
    exist_ok=True,
)
print("Done. Best weights:", 'runs/colony/weights/best.pt')


## 7. See how it did

Two things to look at:
- **The metrics** printed above (mAP50 is the headline — higher is better).
- **Actual detections** on validation plates below. This is the real test:
  do the boxes land on colonies? Trust your eyes here over any single number.


In [ ]:
from ultralytics import YOLO
import glob
from IPython.display import Image, display

best = YOLO('runs/colony/weights/best.pt')

# Run on the held-out validation plates and show a few.
val_imgs = glob.glob('dataset/val/images/*.jpg')
pred = best.predict(val_imgs, save=True, conf=0.25,
                    project='runs', name='preds', exist_ok=True)

shown = glob.glob('runs/preds/*.jpg')[:6]
for p in shown:
    display(Image(filename=p, width=520))


## 8. Count with the model

This is what you'll actually use: point it at a plate, get a count and the
marked-up image. The `conf` threshold trades off missed colonies vs false
positives — tune it to match your eye.


In [ ]:
def count_plate(path, conf=0.25):
    res = best.predict(path, conf=conf, save=True,
                       project='runs', name='count', exist_ok=True)[0]
    n = len(res.boxes)
    print(f"{os.path.basename(path)}: {n} colonies (conf>={conf})")
    out = glob.glob('runs/count/*.jpg')
    if out:
        display(Image(filename=sorted(out)[-1], width=520))
    return n

# Example on a validation plate:
if val_imgs:
    count_plate(val_imgs[0])


## 9. Download the trained model

Grab `best.pt` — this is your model. Later we'll wire it into the counter app
so it replaces the classical CV engine.


In [ ]:
from google.colab import files
files.download('runs/colony/weights/best.pt')
